# Week 5 - Q5: What is a customer worth over time?
**Question.** Government leads on revenue through order count, not order size. What does a customer-lifetime-value view show?

**Approach.** Build one row per customer (500 customers) from order dates, then compare customer types, measure how concentrated value is, and check whether a customer's value is a stable trait. Lifetime value is defined as **observed gross profit over 2019 to 2024** on Delivered orders. It is a measurement, not a forecast: the data holds a single price list and no churn events, so projecting forward would be an assumption, not evidence.

**Definitions (same as Weeks 3 and 4).** Net revenue = `line_total` (ex-GST); gross profit = `line_profit = quantity x (unit_price - unit_cost)`; Delivered orders only. Customer dates come from **order dates**, not the customer master (`customer_since` and `last_purchase_date` are shown to be unreliable in Part 7). Built on the corrected Week 2 merged datasets.

In [1]:
import pandas as pd, numpy as np, os, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SURF,INK,INK2,GRID,BLUE,ORANGE,AQUA,MUTED="#fcfcfb","#0b0b0b","#52514e","#e4e3df","#2a78d6","#eb6834","#1baf7a","#b9b8b2"
def find_dir(env_var, marker, rels):
    """Environment variable first, then this folder and every parent folder, using the repository layout."""
    if os.environ.get(env_var) and (Path(os.environ[env_var]) / marker).exists():
        return Path(os.environ[env_var])
    for base in [Path.cwd(), *Path.cwd().parents]:
        for rel in rels:
            if (base / rel / marker).exists():
                return base / rel
    return None
DATA_DIR = find_dir("CADETX_MERGED", "sales_orders_full.csv", ["merged", "data/merged", "week-02/reports/week_02/merged", "reports/week_02/merged"])
assert DATA_DIR is not None, "Could not find the merged datasets. Run from inside the repository, put a merged/ folder next to the notebook, or set the CADETX_MERGED environment variable to the folder."
RAW_DIR = find_dir("CADETX_RAW", "customers.csv", ["raw_data", "data/raw_data", "week-01/data"])
assert RAW_DIR is not None, "Could not find customers.csv. Run from inside the repository, or set the CADETX_RAW environment variable to the folder holding the raw tables."
OUT=Path("q5_outputs"); OUT.mkdir(exist_ok=True); CH=Path("q5_charts"); CH.mkdir(exist_ok=True)
SEED=42; NBOOT=5000
checks=[]
def check(name,expected,actual,ok):
    checks.append(dict(Check_Item=name,Expected=expected,Actual=actual,Status="PASS" if ok else "FAIL")); assert ok,f"{name}: expected {expected}, got {actual}"
def holm(p):
    p=np.asarray(p,float); m=len(p); o=np.argsort(p); adj=np.empty(m); run=0
    for r,i in enumerate(o): run=max(run,(m-r)*p[i]); adj[i]=min(1,run)
    return adj
def eps2(kw,n,k): return max((kw.statistic-k+1)/(n-k),0.0)
print("Merged datasets:",DATA_DIR.resolve()); print("Raw tables:",RAW_DIR.resolve())

Merged datasets: <repository>/week-02/reports/week_02/merged
Raw tables: <repository>/raw_data


## Part 0. Load and integrity checks

In [2]:
lines=pd.read_csv(DATA_DIR/"sales_orders_full.csv",low_memory=False,parse_dates=["order_date","delivery_date"])
check("sales lines",130402,len(lines),len(lines)==130402)
dl=lines[lines.order_status=="Delivered"].copy()
net=dl.line_total.sum(); gp=dl.line_profit.sum()
check("net revenue (Delivered, ex-GST) matches Week 3, Rs 22.96B",22.96,round(net/1e9,2),round(net/1e9,2)==22.96)
check("gross margin is 21.6% (Week 3)",21.6,round(gp/net*100,1),round(gp/net*100,1)==21.6)
recalc=(dl.quantity*(dl.unit_price-dl.unit_cost)).sum()
check("line_profit equals quantity x (price - cost)",0,round(float(recalc-gp),2),abs(recalc-gp)<1)
check("500 customers have Delivered orders",500,dl.customer_id.nunique(),dl.customer_id.nunique()==500)
check("each customer has one type",1,int(dl.groupby("customer_id").customer_type.nunique().max()),dl.groupby("customer_id").customer_type.nunique().max()==1)
print(f"Gross profit Rs {gp/1e9:.2f}B on net revenue Rs {net/1e9:.2f}B ({gp/net*100:.1f}% margin)")

Gross profit Rs 4.95B on net revenue Rs 22.96B (21.6% margin)


## Part 1. One row per customer

In [3]:
od=dl.drop_duplicates("so_id").sort_values(["customer_id","order_date"])
cu=dl.groupby(["customer_id","customer_type","industry_segment","customer_region","customer_rating","customer_payment_terms"]).agg(net_revenue=("line_total","sum"),gross_profit=("line_profit","sum"),orders=("so_id","nunique"),first_order=("order_date","min"),last_order=("order_date","max")).reset_index()
cu["avg_order_value"]=cu.net_revenue/cu.orders
cu["tenure_months"]=(cu.last_order-cu.first_order).dt.days/30.44
cu["orders_per_year"]=cu.orders/6
gap=od.groupby("customer_id").order_date.diff().dt.days; cu["median_gap_days"]=cu.customer_id.map(gap.groupby(od.customer_id).median())
check("customer revenue sums to total",0,round(float(cu.net_revenue.sum()-net),2),abs(cu.net_revenue.sum()-net)<1)
check("customer gross profit sums to total",0,round(float(cu.gross_profit.sum()-gp),2),abs(cu.gross_profit.sum()-gp)<1)
cu.assign(first_order=cu.first_order.dt.date,last_order=cu.last_order.dt.date).round(2).to_csv(OUT/"q5_customer_value_table.csv",index=False)
d=cu[["net_revenue","gross_profit","orders","avg_order_value","tenure_months","median_gap_days"]].describe().T.round(1)
d.to_csv(OUT/"q5_customer_distribution.csv"); print(d.to_string())

                 count        mean         std         min         25%         50%         75%         max
net_revenue      500.0  45917974.9  10298204.4  17721530.0  38446610.0  45321980.0  52438192.5  84966970.0
gross_profit     500.0   9900013.5   2156437.0   4003500.0   8404192.5   9741755.0  11197892.5  18179070.0
orders           500.0        36.1         6.0        21.0        32.0        36.0        40.0        56.0
avg_order_value  500.0   1273690.2    197849.0    738397.1   1131625.4   1269734.3   1395525.8   1955790.7
tenure_months    500.0        67.9         2.8        56.5        66.6        68.5        69.9        71.9
median_gap_days  500.0        42.6        10.1        20.0        35.4        41.0        49.0        81.0


## Part 2. Government leads on orders. Why?
Is Government ahead because each Government customer orders more, or because there are more Government customers?

In [4]:
g=cu.groupby("customer_type").agg(customers=("customer_id","count"),delivered_orders=("orders","sum"),orders_per_customer=("orders","mean"),mean_gross_profit=("gross_profit","mean"),median_gross_profit=("gross_profit","median"),avg_order_value=("avg_order_value","mean"),net_revenue=("net_revenue","sum"))
g["customer_share_pct"]=g.customers/g.customers.sum()*100; g["order_share_pct"]=g.delivered_orders/g.delivered_orders.sum()*100; g["revenue_share_pct"]=g.net_revenue/g.net_revenue.sum()*100
g=g.sort_values("delivered_orders",ascending=False); show=g.copy(); show["net_revenue"]=show.net_revenue/1e6
show.round(2).to_csv(OUT/"q5_by_customer_type.csv"); print(show.round(2).to_string())
gov=g.loc["Government"]
print(f"\nGovernment: {gov.customer_share_pct:.1f}% of customers, {gov.order_share_pct:.1f}% of orders, {gov.revenue_share_pct:.1f}% of revenue; {gov.orders_per_customer:.1f} orders per customer (all types {cu.orders.mean():.1f}); average order Rs {gov.avg_order_value/1e6:.2f}M (lowest of the five)")
check("Government has the most delivered orders (4,075)",4075,int(gov.delivered_orders),g.delivered_orders.idxmax()=="Government" and int(gov.delivered_orders)==4075)
check("Government has the most customers",True,g.customers.idxmax(),g.customers.idxmax()=="Government")
check("Government's order share is within one point of its customer share",'<1',round(abs(gov.order_share_pct-gov.customer_share_pct),2),abs(gov.order_share_pct-gov.customer_share_pct)<1)
check("Government has the lowest average order value",True,g.avg_order_value.idxmin(),g.avg_order_value.idxmin()=="Government")

               customers  delivered_orders  orders_per_customer  mean_gross_profit  median_gross_profit  avg_order_value  net_revenue  customer_share_pct  order_share_pct  revenue_share_pct
customer_type                                                                                                                                                                                
Government           114              4075                35.75         9565416.75            9402855.0       1240333.05      5049.34                22.8            22.60              21.99
Corporate            104              3672                35.31         9754548.85            9674470.0       1284621.70      4717.36                20.8            20.36              20.55
Dealer                99              3633                36.70        10205306.87           10158830.0       1288902.27      4683.09                19.8            20.15              20.40
Fleet Owner           95              3457        

## Part 3. Do customer types differ in lifetime value?

In [5]:
types=sorted(cu.customer_type.unique()); k=len(types); n=len(cu); res=[]
for lab,col in [("Gross profit per customer (lifetime value)","gross_profit"),("Orders per customer","orders"),("Average order value","avg_order_value"),("Net revenue per customer","net_revenue")]:
    kw=stats.kruskal(*[cu.loc[cu.customer_type==t,col] for t in types]); res.append(dict(measure=lab,kruskal_h=kw.statistic,p_value=kw.pvalue,epsilon_squared=eps2(kw,n,k)))
R=pd.DataFrame(res); R.round(4).to_csv(OUT/"q5_type_tests.csv",index=False); print(R.round(3).to_string(index=False))
pairs=[]
for i in range(k):
    for j in range(i+1,k):
        a=cu.loc[cu.customer_type==types[i],"gross_profit"]; b=cu.loc[cu.customer_type==types[j],"gross_profit"]; pairs.append(dict(type_a=types[i],type_b=types[j],mean_diff=a.mean()-b.mean(),p_value=stats.mannwhitneyu(a,b).pvalue))
Pp=pd.DataFrame(pairs); Pp["p_holm_adjusted"]=holm(Pp.p_value); Pp.round(4).to_csv(OUT/"q5_type_pairwise.csv",index=False)
rng=np.random.default_rng(SEED); hi_t=g.mean_gross_profit.idxmax(); lo_t=g.mean_gross_profit.idxmin()
a=cu.loc[cu.customer_type==hi_t,"gross_profit"].values; b=cu.loc[cu.customer_type==lo_t,"gross_profit"].values
boot=np.array([rng.choice(a,len(a)).mean()-rng.choice(b,len(b)).mean() for _ in range(NBOOT)]); lo,hi=np.percentile(boot,[2.5,97.5])
print(f"\nWidest gap in mean lifetime value: {hi_t} (Rs {a.mean()/1e6:.2f}M) minus {lo_t} (Rs {b.mean()/1e6:.2f}M) = Rs {(a.mean()-b.mean())/1e6:.2f}M ({(a.mean()/b.mean()-1)*100:.1f}%)")
print(f"A plain bootstrap interval for that pair is Rs {lo/1e6:.2f}M to Rs {hi/1e6:.2f}M, but it OVERSTATES the evidence: this pair was picked as the largest of 10 pairs after looking. The fair test is whether the spread of the five type means is bigger than chance produces.")
vals=cu.gross_profit.values; labs=cu.customer_type.values; obs_range=g.mean_gross_profit.max()-g.mean_gross_profit.min()
rng=np.random.default_rng(SEED); null=np.empty(NBOOT)
for i in range(NBOOT):
    sh=rng.permutation(labs); mm=pd.Series(vals).groupby(sh).mean(); null[i]=mm.max()-mm.min()
p_range=(1+(null>=obs_range).sum())/(NBOOT+1)
print(f"Spread between the best and worst type mean: Rs {obs_range/1e6:.2f}M. Under random labels the spread is typically Rs {np.median(null)/1e6:.2f}M (95th percentile Rs {np.percentile(null,95)/1e6:.2f}M). Permutation p = {p_range:.2f}")
print(f"Smallest adjusted pairwise p = {Pp.p_holm_adjusted.min():.2f}")
pd.DataFrame([[hi_t,lo_t,a.mean(),b.mean(),a.mean()-b.mean(),lo,hi,obs_range,np.median(null),np.percentile(null,95),p_range]],columns=["highest_type","lowest_type","highest_mean","lowest_mean","difference","naive_boot_low","naive_boot_high","range_of_type_means","null_median_range","null_95th_range","permutation_p"]).round(3).to_csv(OUT/"q5_widest_gap.csv",index=False)
check("lifetime value does not differ by customer type (p > 0.05)",">0.05",round(R.p_value.iloc[0],3),R.p_value.iloc[0]>0.05)
check("customer-type effect on lifetime value is negligible (epsilon-squared < 0.01)","<0.01",round(R.epsilon_squared.iloc[0],4),R.epsilon_squared.iloc[0]<0.01)
check("no pair of types differs after Holm adjustment",">0.05",round(Pp.p_holm_adjusted.min(),3),Pp.p_holm_adjusted.min()>0.05)
check("the spread of the five type means is within chance (permutation p > 0.05)",">0.05",round(p_range,3),p_range>0.05)

                                   measure  kruskal_h  p_value  epsilon_squared
Gross profit per customer (lifetime value)      3.956    0.412              0.0
                       Orders per customer      1.861    0.761              0.0
                       Average order value      3.348    0.501              0.0
                  Net revenue per customer      3.815    0.432              0.0



Widest gap in mean lifetime value: Dealer (Rs 10.21M) minus Government (Rs 9.57M) = Rs 0.64M (6.7%)
A plain bootstrap interval for that pair is Rs 0.06M to Rs 1.22M, but it OVERSTATES the evidence: this pair was picked as the largest of 10 pairs after looking. The fair test is whether the spread of the five type means is bigger than chance produces.


Spread between the best and worst type mean: Rs 0.64M. Under random labels the spread is typically Rs 0.50M (95th percentile Rs 0.82M). Permutation p = 0.23
Smallest adjusted pairwise p = 0.57


## Part 3b. Other ways to segment customers
Region, industry, rating and payment terms, adjusted together with customer type.

In [6]:
seg=[]
for lab,col in [("Customer type","customer_type"),("Customer region","customer_region"),("Industry segment","industry_segment"),("Customer rating","customer_rating"),("Customer payment terms","customer_payment_terms")]:
    grp=[x.gross_profit.values for _,x in cu.groupby(col)]; kw=stats.kruskal(*grp); seg.append(dict(dimension=lab,segments=len(grp),smallest_segment_customers=int(min(len(x) for x in grp)),p_value=kw.pvalue,epsilon_squared=eps2(kw,n,len(grp))))
Sg=pd.DataFrame(seg); Sg["p_holm_adjusted"]=holm(Sg.p_value); Sg.round(4).to_csv(OUT/"q5_segment_tests.csv",index=False); print(Sg.round(3).to_string(index=False))
ind=cu.groupby("industry_segment").agg(customers=("customer_id","count"),mean_gross_profit=("gross_profit","mean"),median_gross_profit=("gross_profit","median")).sort_values("mean_gross_profit",ascending=False)
ind.round(0).to_csv(OUT/"q5_industry_detail.csv"); print("\nIndustry segment is the closest to a signal (small effect, not significant after adjustment):"); print(ind.assign(mean_gross_profit=ind.mean_gross_profit/1e6,median_gross_profit=ind.median_gross_profit/1e6).round(2).rename(columns={"mean_gross_profit":"mean_gp_M","median_gross_profit":"median_gp_M"}).to_string())
print(f"Highest vs lowest industry mean: {ind.mean_gross_profit.max()/ind.mean_gross_profit.min()*100-100:.1f}% apart. Raw p = {Sg.loc[Sg.dimension=='Industry segment','p_value'].iloc[0]:.3f}, adjusted for 5 cuts = {Sg.loc[Sg.dimension=='Industry segment','p_holm_adjusted'].iloc[0]:.3f}. Part 3c tests whether it is real.")
check("no customer segment (5 cuts) differs in lifetime value after adjustment (industry is closest)",">0.05",round(Sg.p_holm_adjusted.min(),3),Sg.p_holm_adjusted.min()>0.05)

             dimension  segments  smallest_segment_customers  p_value  epsilon_squared  p_holm_adjusted
         Customer type         5                          88    0.412            0.000            1.000
       Customer region         6                          11    0.269            0.003            1.000
      Industry segment         6                          67    0.012            0.020            0.059
       Customer rating         5                          88    0.629            0.000            1.000
Customer payment terms         5                          87    0.985            0.000            1.000

Industry segment is the closest to a signal (small effect, not significant after adjustment):
                  customers  mean_gp_M  median_gp_M
industry_segment                                   
Manufacturing            70      10.46        10.44
Rental                   67      10.26        10.17
Infrastructure           79      10.07         9.85
Logistics            

## Part 3c. Can the industry-segment lead be settled?
Industry segment came close to a signal. Two tests: (1) a **permutation test** on the spread of the six industry means (is the gap larger than shuffled labels produce?), and (2) a **split-half replication**: if the industry pattern is real, the industry ranking from 2019-2021 should repeat in 2022-2024. **Caveat:** both halves use the same 500 customers, so repeating shows the pattern is stable inside this data, not that it will hold for new customers.

In [7]:
vals=cu.gross_profit.values; labs=cu.industry_segment.values; obs_rng=ind.mean_gross_profit.max()-ind.mean_gross_profit.min()
rng=np.random.default_rng(SEED); nullr=np.empty(NBOOT)
for i in range(NBOOT):
    mm=pd.Series(vals).groupby(rng.permutation(labs)).mean(); nullr[i]=mm.max()-mm.min()
p_ind=(1+(nullr>=obs_rng).sum())/(NBOOT+1)
ie=dl[dl.order_date.dt.year<=2021].groupby(["customer_id","industry_segment"]).line_profit.sum().reset_index().groupby("industry_segment").line_profit.mean()
il=dl[dl.order_date.dt.year>=2022].groupby(["customer_id","industry_segment"]).line_profit.sum().reset_index().groupby("industry_segment").line_profit.mean()
rho_i,p_rho_i=stats.spearmanr(ie.sort_index(),il.sort_index())
rep=pd.DataFrame({"early_2019_2021":ie,"late_2022_2024":il}); rep["early_rank"]=rep.early_2019_2021.rank(ascending=False).astype(int); rep["late_rank"]=rep.late_2022_2024.rank(ascending=False).astype(int)
rep.round(0).to_csv(OUT/"q5_industry_replication.csv")
print(f"Spread between best and worst industry mean: Rs {obs_rng/1e6:.2f}M. Under shuffled labels the typical spread is Rs {np.median(nullr)/1e6:.2f}M (95th percentile Rs {np.percentile(nullr,95)/1e6:.2f}M); permutation p = {p_ind:.3f}")
print(f"Split-half replication: rank correlation of the six industry means between 2019-2021 and 2022-2024 = {rho_i:.2f} (n = 6 industries, p = {p_rho_i:.2f})")
print(rep.assign(early_2019_2021=rep.early_2019_2021/1e6,late_2022_2024=rep.late_2022_2024/1e6).round(2).to_string())
print("Reading: a real industry effect should keep its ranking in the later half. A low or negative correlation means the early gap does not repeat.")
pd.DataFrame([["industry spread (Rs)",obs_rng],["permutation p",p_ind],["null median spread (Rs)",np.median(nullr)],["split-half Spearman rho (6 industries)",rho_i],["split-half p",p_rho_i]],columns=["measure","value"]).round(4).to_csv(OUT/"q5_industry_tests.csv",index=False)
ic=cu.groupby("industry_segment").agg(customers=("customer_id","count"),orders_per_customer=("orders","mean"),avg_order_value=("avg_order_value","mean"),gross_margin_pct=("gross_profit",lambda s:0)).drop(columns="gross_margin_pct")
ic["gross_margin_pct"]=(cu.groupby("industry_segment").gross_profit.sum()/cu.groupby("industry_segment").net_revenue.sum()*100)
cu["margin_pct"]=cu.gross_profit/cu.net_revenue*100
kw_n=stats.kruskal(*[x.orders.values for _,x in cu.groupby("industry_segment")]); kw_a=stats.kruskal(*[x.avg_order_value.values for _,x in cu.groupby("industry_segment")]); kw_m=stats.kruskal(*[x.margin_pct.values for _,x in cu.groupby("industry_segment")])
ic.round(2).sort_values("orders_per_customer",ascending=False).to_csv(OUT/"q5_industry_drivers.csv"); print("\nWhat drives the industry difference?"); print(ic.round(2).sort_values("orders_per_customer",ascending=False).to_string())
print(f"Kruskal-Wallis by industry: orders per customer p = {kw_n.pvalue:.3f}; average order value p = {kw_a.pvalue:.2f}; gross margin p = {kw_m.pvalue:.2f}. The gap comes from how often customers order, not order size or margin.")
IND_RAW_SIG=bool(p_ind<0.05); IND_ADJ_NS=bool(Sg.loc[Sg.dimension=="Industry segment","p_holm_adjusted"].iloc[0]>0.05)
check("industry is an exploratory lead, not a confirmed finding (raw permutation p < 0.05 but not significant once 5 cuts are adjusted)",True,(round(p_ind,3),round(float(Sg.loc[Sg.dimension=="Industry segment","p_holm_adjusted"].iloc[0]),3)),IND_RAW_SIG and IND_ADJ_NS)
check("the industry ranking repeats across the two halves of the period (rank correlation > 0.6)",">0.6",round(rho_i,2),rho_i>0.6)
check("the industry difference is driven by order frequency, not margin (orders p < 0.05, margin p > 0.05)",True,(round(kw_n.pvalue,3),round(kw_m.pvalue,2)),kw_n.pvalue<0.05 and kw_m.pvalue>0.05)

Spread between best and worst industry mean: Rs 1.12M. Under shuffled labels the typical spread is Rs 0.59M (95th percentile Rs 0.96M); permutation p = 0.014
Split-half replication: rank correlation of the six industry means between 2019-2021 and 2022-2024 = 0.94 (n = 6 industries, p = 0.00)
                  early_2019_2021  late_2022_2024  early_rank  late_rank
industry_segment                                                        
Construction                 4.85            4.82           5          5
Infrastructure               5.02            5.06           3          3
Logistics                    5.00            4.88           4          4
Manufacturing                5.40            5.06           1          2
Mining                       4.84            4.50           6          6
Rental                       5.15            5.11           2          1
Reading: a real industry effect should keep its ranking in the later half. A low or negative correlation means the early ga

## Part 4. How concentrated is customer value?

In [8]:
x=np.sort(cu.gross_profit.values); N=len(x); cs=np.cumsum(x[::-1])/x.sum()
gini=(2*np.arange(1,N+1)@x)/(N*x.sum())-(N+1)/N
top1=cs[0]*100; top5=cs[4]*100; top10p=cs[49]*100; top20p=cs[99]*100
p10,p50,p90=np.percentile(x,[10,50,90])
print(f"Lifetime gross profit per customer: median Rs {p50/1e6:.2f}M, 10th percentile Rs {p10/1e6:.2f}M, 90th percentile Rs {p90/1e6:.2f}M (90th/10th = {p90/p10:.1f}x)")
print(f"Largest customer {top1:.2f}% of gross profit; top 5 {top5:.2f}%; top 10% of customers {top10p:.1f}%; top 20% {top20p:.1f}%; Gini {gini:.3f} (0 = perfectly equal, 1 = one customer owns everything)")
pd.DataFrame([["largest customer share (%)",top1],["top 5 customers share (%)",top5],["top 10% of customers share (%)",top10p],["top 20% of customers share (%)",top20p],["Gini coefficient",gini],["10th percentile (Rs)",p10],["median (Rs)",p50],["90th percentile (Rs)",p90],["90th / 10th percentile",p90/p10]],columns=["measure","value"]).round(4).to_csv(OUT/"q5_concentration.csv",index=False)
check("no customer dominates (largest < 1% of gross profit)","<1",round(top1,2),top1<1)
check("value is spread evenly: top 10% of customers hold under 20% (the 80/20 rule does not apply)","<20",round(top10p,1),top10p<20)
rng=np.random.default_rng(SEED); bm=np.array([rng.choice(x,N).mean() for _ in range(NBOOT)]); mlo,mhi=np.percentile(bm,[2.5,97.5])
print(f"Mean lifetime gross profit per customer Rs {x.mean()/1e6:.2f}M, bootstrap 95% interval Rs {mlo/1e6:.2f}M to Rs {mhi/1e6:.2f}M")
pd.DataFrame([["mean lifetime gross profit (Rs)",x.mean()],["bootstrap 95% low",mlo],["bootstrap 95% high",mhi]],columns=["measure","value"]).round(0).to_csv(OUT/"q5_mean_value_interval.csv",index=False)
check("Gini is low (< 0.25)","<0.25",round(gini,3),gini<0.25)

Lifetime gross profit per customer: median Rs 9.74M, 10th percentile Rs 7.32M, 90th percentile Rs 12.66M (90th/10th = 1.7x)
Largest customer 0.37% of gross profit; top 5 1.66%; top 10% of customers 14.2%; top 20% 26.5%; Gini 0.121 (0 = perfectly equal, 1 = one customer owns everything)
Mean lifetime gross profit per customer Rs 9.90M, bootstrap 95% interval Rs 9.71M to Rs 10.08M


## Part 5. Lifetime shape: tenure, activity and frequency

In [9]:
yrs=dl.assign(y=dl.order_date.dt.year).groupby("y").agg(active_customers=("customer_id","nunique"),gross_profit=("line_profit","sum"),orders=("so_id","nunique"))
yrs["gross_profit_per_active_customer"]=yrs.gross_profit/yrs.active_customers; yrs["orders_per_active_customer"]=yrs.orders/yrs.active_customers
yrs.round(2).to_csv(OUT/"q5_yearly_activity.csv"); print(yrs.round(1).to_string())
fo=cu.first_order.dt.year.value_counts().sort_index(); lo_=cu.last_order.dt.year.value_counts().sort_index()
print(f"\nFirst order year: {fo.to_dict()}  |  last order year: {lo_.to_dict()}")
print(f"Tenure {cu.tenure_months.min():.0f} to {cu.tenure_months.max():.0f} months of a possible 72; median gap between orders {cu.median_gap_days.median():.0f} days")
print(f"Every customer ordered in both 2019 and 2024 and at least {int(yrs.active_customers.min())} of 500 were active in every year, so the data shows no new customers and no lost customers. Lifetime differences come from how much a customer buys, not how long they stay.")
chg=(yrs.gross_profit_per_active_customer.iloc[-1]/yrs.gross_profit_per_active_customer.iloc[0]-1)*100
print(f"Gross profit per active customer moved {chg:.1f}% from 2019 to 2024 (the same fall seen in Q1)")
check("all 500 customers placed their first Delivered order in 2019",500,int(fo.get(2019,0)),int(fo.get(2019,0))==500)
check("all 500 customers placed a Delivered order in 2024",500,int(lo_.get(2024,0)),int(lo_.get(2024,0))==500)
check("at least 99% of customers (495 of 500) are active in every year",">=495",int(yrs.active_customers.min()),yrs.active_customers.min()>=495)

      active_customers  gross_profit  orders  gross_profit_per_active_customer  orders_per_active_customer
y                                                                                                         
2019               500     834440040    3031                         1668880.1                         6.1
2020               500     845098840    3038                         1690197.7                         6.1
2021               498     831184120    3038                         1669044.4                         6.1
2022               498     821097090    2948                         1648789.3                         5.9
2023               500     824558720    3022                         1649117.4                         6.0
2024               500     793627930    2956                         1587255.9                         5.9

First order year: {2019: 500}  |  last order year: {2024: 500}
Tenure 57 to 72 months of a possible 72; median gap between orders 41 days
Every

## Part 5b. Is a customer's value a stable trait?
If some customers are inherently more valuable, those who earned more in 2019 to 2021 should earn more in 2022 to 2024. Spearman rank correlation, 500 customers.

In [10]:
early=dl[dl.order_date.dt.year<=2021].groupby("customer_id").line_profit.sum(); late=dl[dl.order_date.dt.year>=2022].groupby("customer_id").line_profit.sum()
both=pd.concat([early.rename("early"),late.rename("late")],axis=1).dropna()
rho,prho=stats.spearmanr(both.early,both.late)
rng=np.random.default_rng(SEED); bs=[stats.spearmanr(*both.sample(len(both),replace=True,random_state=int(rng.integers(1e9))).T.values)[0] for _ in range(1000)]; rl,rh=np.percentile(bs,[2.5,97.5])
print(f"Early (2019-2021) vs late (2022-2024) gross profit per customer: Spearman rho = {rho:.3f}, p = {prho:.2f}, bootstrap 95% interval [{rl:.2f}, {rh:.2f}]")
pd.DataFrame([["Spearman rho, early vs late gross profit",rho],["p-value",prho],["bootstrap 95% low (1,000 resamples)",rl],["bootstrap 95% high",rh]],columns=["measure","value"]).round(4).to_csv(OUT/"q5_value_persistence.csv",index=False)
check("value persistence is weak (|rho| < 0.2)","<0.2",round(abs(rho),3),abs(rho)<0.2)

Early (2019-2021) vs late (2022-2024) gross profit per customer: Spearman rho = -0.025, p = 0.58, bootstrap 95% interval [-0.11, 0.07]


## Part 6. Value before and after unpaid balances (sensitivity)
19.8% of invoiced value is outstanding and most of it is more than a year overdue (Week 3). Each Delivered order is matched to its single invoice, and the order's gross profit is reduced by that invoice's unpaid share (`outstanding_amount / grand_total`); the results are then summed per customer. This is finer than a customer-level average, but it is still an **indicative sensitivity, not a measured collected profit**: it assumes profit is lost in proportion to the unpaid share, and an unpaid invoice may still be collected.

In [11]:
inv=pd.read_csv(DATA_DIR/"invoices_with_payments.csv",low_memory=False)
check("one invoice per Delivered order",18033,len(inv),len(inv)==18033)
tot_out=inv.outstanding_amount.sum(); tot_inv=inv.grand_total.sum()
over1y=inv.loc[inv.days_overdue>365,"outstanding_amount"].sum()/tot_out*100
og=dl.groupby("so_id").agg(customer_id=("customer_id","first"),gp=("line_profit","sum"),rev=("line_total","sum")).reset_index()
oi=og.merge(inv[["so_id","grand_total","outstanding_amount"]],on="so_id",how="left")
check("every Delivered order is matched to exactly one invoice",18033,int(oi.grand_total.notna().sum()),int(oi.grand_total.notna().sum())==18033 and len(oi)==18033)
oi["share"]=oi.outstanding_amount/oi.grand_total; oi["gp_after"]=oi.gp*(1-oi.share)
cb=oi.groupby("customer_id").agg(gross_profit_o=("gp","sum"),gp_after_outstanding=("gp_after","sum"),invoiced=("grand_total","sum"),outstanding=("outstanding_amount","sum"))
cu2=cu.merge(cb[["gp_after_outstanding","invoiced","outstanding"]],left_on="customer_id",right_index=True); cu2["outstanding_share"]=cu2.outstanding/cu2.invoiced
print(f"Outstanding Rs {tot_out/1e9:.2f}B of Rs {tot_inv/1e9:.2f}B invoiced ({tot_out/tot_inv*100:.1f}%); {over1y:.0f}% of the outstanding amount is more than a year overdue")
t=cu2.groupby("customer_type").agg(mean_gp=("gross_profit","mean"),mean_gp_after=("gp_after_outstanding","mean"),mean_outstanding_share=("outstanding_share","mean")); t["mean_outstanding_share"]*=100
kwo=stats.kruskal(*[x.outstanding_share.values for _,x in cu2.groupby("customer_type")]); kwa=stats.kruskal(*[x.gp_after_outstanding.values for _,x in cu2.groupby("customer_type")])
print(t.round(1).to_string()); print(f"\nOutstanding share by type: Kruskal-Wallis p = {kwo.pvalue:.2f}; lifetime value after outstanding by type: p = {kwa.pvalue:.2f}")
t.round(2).to_csv(OUT/"q5_value_after_outstanding.csv"); cu2[["customer_id","customer_type","gross_profit","outstanding_share","gp_after_outstanding"]].round(4).to_csv(OUT/"q5_value_after_outstanding_by_customer.csv",index=False)
check("outstanding share of invoiced value is about 19.8% (Week 3)",19.8,round(tot_out/tot_inv*100,1),round(tot_out/tot_inv*100,1)==19.8)
check("86% of outstanding is over a year overdue (Week 3)",86,round(over1y),round(over1y)==86)
check("outstanding share does not differ by customer type (p > 0.05)",">0.05",round(kwo.pvalue,3),kwo.pvalue>0.05)
check("the ranking message holds after the adjustment: no type differs (p > 0.05)",">0.05",round(kwa.pvalue,3),kwa.pvalue>0.05)

Outstanding Rs 5.80B of Rs 29.25B invoiced (19.8%); 86% of the outstanding amount is more than a year overdue
                  mean_gp  mean_gp_after  mean_outstanding_share
customer_type                                                   
Corporate       9754548.8      7487516.5                    21.3
Dealer         10205306.9      8028758.6                    19.7
Fleet Owner    10008847.4      7786942.2                    19.7
Government      9565416.8      7450642.8                    19.6
Retail         10044434.9      7902719.7                    19.3

Outstanding share by type: Kruskal-Wallis p = 0.27; lifetime value after outstanding by type: p = 0.23


## Part 7. Why customer master dates are not used (Q13 preview)

In [12]:
cm=pd.read_csv(RAW_DIR/"customers.csv",parse_dates=["customer_since","last_purchase_date"])
m=cu.merge(cm[["customer_id","customer_since","last_purchase_date","total_purchase_value"]],on="customer_id")
since_after=(m.customer_since>m.first_order).mean()*100; last_before=(m.last_purchase_date<m.last_order).mean()*100
tpv=(m.total_purchase_value.sum()/net)
print(f"customer_since is AFTER the customer's first Delivered order for {since_after:.0f}% of customers (every customer's first order is in 2019).")
print(f"last_purchase_date is BEFORE the customer's last Delivered order for {last_before:.0f}% of customers.")
print(f"Sum of total_purchase_value in the master is {tpv*100:.1f}% of Delivered net revenue, and its customer-level correlation with revenue is {m.total_purchase_value.corr(m.net_revenue):.2f}.")
pd.DataFrame([["customers with customer_since after first order (%)",since_after],["customers with last_purchase_date before last order (%)",last_before],["master total_purchase_value / Delivered net revenue (%)",tpv*100],["correlation of master total_purchase_value with order revenue",m.total_purchase_value.corr(m.net_revenue)]],columns=["measure","value"]).round(3).to_csv(OUT/"q5_master_date_check.csv",index=False)
check("customer master dates conflict with order history for a large share of customers (> 25%)",">25",round(since_after,1),since_after>25)

customer_since is AFTER the customer's first Delivered order for 58% of customers (every customer's first order is in 2019).
last_purchase_date is BEFORE the customer's last Delivered order for 93% of customers.
Sum of total_purchase_value in the master is 5.4% of Delivered net revenue, and its customer-level correlation with revenue is -0.08.


## Charts

In [13]:
order=["Government","Corporate","Dealer","Fleet Owner","Retail"]
fig,ax=plt.subplots(figsize=(8.4,4.6),facecolor=SURF); ax.set_facecolor(SURF)
data=[cu.loc[cu.customer_type==t,"gross_profit"]/1e6 for t in order]
bp=ax.boxplot(data,vert=False,patch_artist=True,widths=0.55,medianprops=dict(color=INK,lw=2),whiskerprops=dict(color=MUTED),capprops=dict(color=MUTED),flierprops=dict(marker="o",markerfacecolor=MUTED,markeredgecolor=MUTED,ms=3))
for b in bp["boxes"]: b.set(facecolor="#cfe0f5",edgecolor=BLUE)
for i,t in enumerate(order,1): ax.text(cu.gross_profit.max()/1e6+0.4,i,f"n={int((cu.customer_type==t).sum())}, mean {cu.loc[cu.customer_type==t,'gross_profit'].mean()/1e6:.1f}M",va="center",fontsize=8.5,color=INK2)
ax.set_yticks(range(1,6)); ax.set_yticklabels(order,color=INK2); ax.invert_yaxis(); ax.set_xlabel("Lifetime gross profit per customer, 2019 to 2024 (Rs million)",color=INK2); ax.set_xlim(0,27)
ax.grid(axis="x",color=GRID,lw=.6); ax.set_axisbelow(True); [s.set_visible(False) for k,s in ax.spines.items() if k in("top","right")]
ax.set_title("Customer types are worth the same over a lifetime",loc="left",color=INK,fontsize=12,fontweight="bold")
ax.text(0.0,-0.17,f"Kruskal-Wallis p = {R.p_value.iloc[0]:.2f}; effect size epsilon-squared = {R.epsilon_squared.iloc[0]:.3f}.",transform=ax.transAxes,fontsize=8.5,color=INK2)
plt.tight_layout(); plt.savefig(CH/"q5_01_lifetime_value_by_type.png",dpi=150,facecolor=SURF); plt.close()

In [14]:
fig,axs=plt.subplots(1,2,figsize=(10.5,4.2),facecolor=SURF)
ax=axs[0]; ax.set_facecolor(SURF); lx=np.concatenate([[0],np.arange(1,N+1)/N*100]); ly=np.concatenate([[0],np.cumsum(x)/x.sum()*100])
ax.plot([0,100],[0,100],color=MUTED,ls="--",lw=1.2,label="Perfect equality"); ax.plot(lx,ly,color=BLUE,lw=2.4,label=f"Actual (Gini {gini:.2f})")
ax.set_xlabel("Customers, poorest to richest (%)",color=INK2); ax.set_ylabel("Cumulative gross profit (%)",color=INK2); ax.legend(frameon=False,loc="upper left"); ax.grid(color=GRID,lw=.6); ax.set_axisbelow(True)
ax.set_title("Value is spread almost evenly",loc="left",fontsize=11,fontweight="bold",color=INK)
ax=axs[1]; ax.set_facecolor(SURF); ax.scatter(both.early/1e6,both.late/1e6,s=14,color=BLUE,alpha=.55,edgecolor="none")
ax.set_xlabel("Gross profit 2019-2021 (Rs million)",color=INK2); ax.set_ylabel("Gross profit 2022-2024 (Rs million)",color=INK2); ax.grid(color=GRID,lw=.6); ax.set_axisbelow(True)
ax.set_title(f"Early value does not predict later value (rho = {rho:.2f})",loc="left",fontsize=11,fontweight="bold",color=INK)
for a_ in axs: [s.set_visible(False) for k,s in a_.spines.items() if k in("top","right")]
plt.tight_layout(); plt.savefig(CH/"q5_02_concentration_and_persistence.png",dpi=150,facecolor=SURF); plt.close()

In [15]:
fig,ax=plt.subplots(figsize=(8.2,4.0),facecolor=SURF); ax.set_facecolor(SURF)
ax.plot(yrs.index,yrs.gross_profit_per_active_customer/1e6,"o-",color=BLUE,lw=2.4,ms=7)
for xx,yy in zip(yrs.index,yrs.gross_profit_per_active_customer/1e6): ax.text(xx,yy+0.025,f"{yy:.2f}M",ha="center",fontsize=9,color=INK2)
ax.set_ylim(1.2,1.8); ax.set_ylabel("Gross profit per active customer (Rs million)",color=INK2); ax.grid(axis="y",color=GRID,lw=.6); ax.set_axisbelow(True)
[s.set_visible(False) for k,s in ax.spines.items() if k in("top","right")]
ax.set_title("Nearly all customers are active every year; value per customer drifts down",loc="left",color=INK,fontsize=11.5,fontweight="bold")
plt.tight_layout(); plt.savefig(CH/"q5_03_value_per_customer_by_year.png",dpi=150,facecolor=SURF); plt.close()

## Q5 answer

In [16]:
print("Q5 ANSWER")
print(f"- Government leads on orders because it has the most customers ({gov.customer_share_pct:.1f}% of customers, {gov.order_share_pct:.1f}% of orders), not because Government customers buy more often ({gov.orders_per_customer:.1f} orders each, in line with all types) or buy bigger orders (the smallest average order).")
print(f"- A customer's lifetime value (observed 2019-2024 gross profit) averages Rs {cu.gross_profit.mean()/1e6:.1f}M (95% interval Rs {mlo/1e6:.1f}M to Rs {mhi/1e6:.1f}M) and does not differ by customer type (p = {R.p_value.iloc[0]:.2f}, effect size {R.epsilon_squared.iloc[0]:.3f}), region, rating or payment terms.")
print(f"- Industry segment is an exploratory lead: Manufacturing and Rental customers are worth about 6-12% more than Construction and Mining. The gap is beyond shuffled labels (permutation p = {p_ind:.3f}) and keeps its ranking in both halves of the period (rank correlation {rho_i:.2f}; same customers in both, so this is stability, not independent confirmation), and it comes from how often customers order (orders per customer p = {kw_n.pvalue:.3f}), not order size or margin. It is not significant once the five segment cuts are adjusted together (p = {Sg.loc[Sg.dimension=='Industry segment','p_holm_adjusted'].iloc[0]:.2f}), so treat it as an exploratory lead to re-test with more data.")
print(f"- Value is evenly spread: the largest customer is {top1:.2f}% of gross profit and the top 10% of customers hold {top10p:.1f}%. There is no 'best customer' group to protect or chase.")
print(f"- A customer's early value does not predict later value (rho = {rho:.2f}), so customer-level value is mostly not a stable trait.")
print("- All 500 customers ordered in both 2019 and 2024, so the data cannot show churn, acquisition or a true customer lifetime. Lifetime value here is observed six-year gross profit, not a forecast.")
print(f"- Adjusting for unpaid balances ({tot_out/tot_inv*100:.1f}% of invoiced value) lowers every type's value by about the same share and does not change the comparison.")

Q5 ANSWER
- Government leads on orders because it has the most customers (22.8% of customers, 22.6% of orders), not because Government customers buy more often (35.7 orders each, in line with all types) or buy bigger orders (the smallest average order).
- A customer's lifetime value (observed 2019-2024 gross profit) averages Rs 9.9M (95% interval Rs 9.7M to Rs 10.1M) and does not differ by customer type (p = 0.41, effect size 0.000), region, rating or payment terms.
- Industry segment is an exploratory lead: Manufacturing and Rental customers are worth about 6-12% more than Construction and Mining. The gap is beyond shuffled labels (permutation p = 0.014) and keeps its ranking in both halves of the period (rank correlation 0.94; same customers in both, so this is stability, not independent confirmation), and it comes from how often customers order (orders per customer p = 0.034), not order size or margin. It is not significant once the five segment cuts are adjusted together (p = 0.06)

## Part 8. Questions for the data owner
1. Are all 500 customers meant to have traded for the full 2019 to 2024 window, or does the extract only keep long-standing customers?
2. Are `customer_since`, `last_purchase_date` and `total_purchase_value` in the customer master maintained, and which should analysis use?
3. Is there a customer acquisition or churn record outside this extract?

## Part 9. Validation summary

In [17]:
V=pd.DataFrame(checks); V.to_csv("validation_report_week05_q5.csv",index=False)
print(f"{(V.Status=='PASS').sum()} of {len(V)} checks pass"); V

34 of 34 checks pass


,Check_Item,Expected,Actual,Status
0,sales lines,130402,130402,PASS
1,"net revenue (Delivered, ex-GST) matches Week 3...",22.96,22.96,PASS
2,gross margin is 21.6% (Week 3),21.6,21.6,PASS
3,line_profit equals quantity x (price - cost),0,0.0,PASS
4,500 customers have Delivered orders,500,500,PASS
5,each customer has one type,1,1,PASS
6,customer revenue sums to total,0,0.0,PASS
7,customer gross profit sums to total,0,0.0,PASS
8,"Government has the most delivered orders (4,075)",4075,4075,PASS
9,Government has the most customers,True,Government,PASS
